In [174]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [175]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TechSagar/sample.csv')

In [176]:
import re

In [177]:
# 1. Merge amt into transaction_amount where it is NaN
df['transaction_amount'] = df['transaction_amount'].fillna(df['amt'])

In [178]:
df


,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,amt
0,TXN-3E3A44E74819,USR0020,7457.251963207814,2024-03-14T11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,Card,140227.79,success,192.96.196.31,NaN
1,TXN-84B8FC890498,USR0043,308.51678134343047,1717321977,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249,NaN
2,TXN-95B71538FFAD,USR0044,₹3159,1710796175,jaipur,Jaipur,NaN,D??,web,Card,168910.49,success,192.73.41.151,NaN
3,TXN-657F2702B5B2,USR0060,5349.73,07/02/2024 03:50:23,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119,NaN
4,TXN-84C8BBF69256,USR0020,2454.5652128035144,"March 13, 2024 11:25 PM",jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,153095.35,success,192.96.196.139,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-09C97C1A8F6A,USR0069,2672.7143498167743,2024-02-26T16:50:59.387741,LKO,lucknow,Electronics,DEV-8657F3E2,web,Wallet,118582.16,success,192.38.168.146,NaN
1443,TXN-C83C897D358E,USR0043,615 INR,1715705824,Pune,Ahmedabad,Clothing,ATO-7D32C693,web,Card,119773.72,success,140.18.71.75,NaN
1444,TXN-A442B398E85E,USR0022,956.784541866445,1708062467,Mumbai,ahmedabad,Entertainment,DEV-54C67087,mobile,Card,73970.71,success,192.62.225.88,NaN
1445,TXN-C0877E9A01B8,USR0048,2332.363563500775,07/02/2024 01:02:50,Chennai,HYD,Healthcare,DEV-D7256DA2,ATM,UPI,11946.20,pending,192.109.213.177,NaN


In [179]:
df.drop(columns=['amt'], inplace=True)

In [180]:
import numpy as np


In [181]:
# Clean currency (Remove ₹, Rs, INR, commas, spaces, etc.)
def clean_currency(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip().lower()
    if val_str in ['(null)', 'null', 'nan', 'none', 'n/a']: return np.nan
    val_str = re.sub(r'[₹|rs|inr|,|\s]', '', val_str)
    return pd.to_numeric(val_str, errors='coerce')

df['transaction_amount'] = df['transaction_amount'].apply(clean_currency)

In [182]:
#1. Clean 'merchant_category'
# Fill missing with 'Others' and convert to Title Case
df['merchant_category'] = df['merchant_category'].fillna('Others').str.title()

In [183]:
df

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14T11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,Card,140227.79,success,192.96.196.31
1,TXN-84B8FC890498,USR0043,308.516781,1717321977,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249
2,TXN-95B71538FFAD,USR0044,3159.000000,1710796175,jaipur,Jaipur,Others,D??,web,Card,168910.49,success,192.73.41.151
3,TXN-657F2702B5B2,USR0060,5349.730000,07/02/2024 03:50:23,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119
4,TXN-84C8BBF69256,USR0020,2454.565213,"March 13, 2024 11:25 PM",jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,153095.35,success,192.96.196.139
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-09C97C1A8F6A,USR0069,2672.714350,2024-02-26T16:50:59.387741,LKO,lucknow,Electronics,DEV-8657F3E2,web,Wallet,118582.16,success,192.38.168.146
1443,TXN-C83C897D358E,USR0043,615.000000,1715705824,Pune,Ahmedabad,Clothing,ATO-7D32C693,web,Card,119773.72,success,140.18.71.75
1444,TXN-A442B398E85E,USR0022,956.784542,1708062467,Mumbai,ahmedabad,Entertainment,DEV-54C67087,mobile,Card,73970.71,success,192.62.225.88
1445,TXN-C0877E9A01B8,USR0048,2332.363564,07/02/2024 01:02:50,Chennai,HYD,Healthcare,DEV-D7256DA2,ATM,UPI,11946.20,pending,192.109.213.177


In [ ]:
# Create soft fraud score instead of hard label
signal_cols = [
    'label_ato','label_cnp','label_new_device_struct',
    'label_bad_ip','label_overdraft','label_micro_probe','label_extreme_amount'
]

df['fraud_score'] = df[signal_cols].sum(axis=1)

# Normalize score (0 to 1)
df['fraud_score_norm'] = df['fraud_score'] / len(signal_cols)

# Use threshold but NOT strict
df['is_fraud_soft'] = (df['fraud_score_norm'] > 0.3).astype(int)

In [184]:
# 2. Clean 'payment_method'
# Standardize to UPPERCASE and fill missing with 'UNKNOWN'
df['payment_method'] = df['payment_method'].fillna('UNKNOWN').str.upper()

In [288]:
df['rolling_avg_3'] = df.groupby('user_id')['transaction_amount'].transform(lambda x: x.rolling(3, min_periods=1).mean())

# Spike detection
df['spike_flag'] = (df['transaction_amount'] > 2 * df['rolling_avg_3']).astype(int)

In [289]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=42)

iso_features = [
    'transaction_amount', 'amount_vs_user_avg',
    'seconds_since_prev', 'txn_count_last_5min',
    'amount_to_balance_ratio'
]

df['iso_score'] = iso.fit_predict(df[iso_features].fillna(0)) # Fillna with 0 for IsolationForest
df['iso_score'] = df['iso_score'].map({1:0, -1:1})  # anomaly = 1

In [185]:
df
y = df['is_fraud_soft']

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14T11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,CARD,140227.79,success,192.96.196.31
1,TXN-84B8FC890498,USR0043,308.516781,1717321977,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249
2,TXN-95B71538FFAD,USR0044,3159.000000,1710796175,jaipur,Jaipur,Others,D??,web,CARD,168910.49,success,192.73.41.151
3,TXN-657F2702B5B2,USR0060,5349.730000,07/02/2024 03:50:23,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119
4,TXN-84C8BBF69256,USR0020,2454.565213,"March 13, 2024 11:25 PM",jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,CARD,153095.35,success,192.96.196.139
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-09C97C1A8F6A,USR0069,2672.714350,2024-02-26T16:50:59.387741,LKO,lucknow,Electronics,DEV-8657F3E2,web,WALLET,118582.16,success,192.38.168.146
1443,TXN-C83C897D358E,USR0043,615.000000,1715705824,Pune,Ahmedabad,Clothing,ATO-7D32C693,web,CARD,119773.72,success,140.18.71.75
1444,TXN-A442B398E85E,USR0022,956.784542,1708062467,Mumbai,ahmedabad,Entertainment,DEV-54C67087,mobile,CARD,73970.71,success,192.62.225.88
1445,TXN-C0877E9A01B8,USR0048,2332.363564,07/02/2024 01:02:50,Chennai,HYD,Healthcare,DEV-D7256DA2,ATM,UPI,11946.20,pending,192.109.213.177


In [186]:
# 3. Clean 'transaction_status'
# Image suggest filtering: Only keep 'success', 'pending', 'failed'
valid_statuses = ['success', 'pending', 'failed']
df = df[df['transaction_status'].str.lower().isin(valid_statuses)]

In [187]:
df

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14T11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,CARD,140227.79,success,192.96.196.31
1,TXN-84B8FC890498,USR0043,308.516781,1717321977,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249
2,TXN-95B71538FFAD,USR0044,3159.000000,1710796175,jaipur,Jaipur,Others,D??,web,CARD,168910.49,success,192.73.41.151
3,TXN-657F2702B5B2,USR0060,5349.730000,07/02/2024 03:50:23,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119
4,TXN-84C8BBF69256,USR0020,2454.565213,"March 13, 2024 11:25 PM",jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,CARD,153095.35,success,192.96.196.139
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-09C97C1A8F6A,USR0069,2672.714350,2024-02-26T16:50:59.387741,LKO,lucknow,Electronics,DEV-8657F3E2,web,WALLET,118582.16,success,192.38.168.146
1443,TXN-C83C897D358E,USR0043,615.000000,1715705824,Pune,Ahmedabad,Clothing,ATO-7D32C693,web,CARD,119773.72,success,140.18.71.75
1444,TXN-A442B398E85E,USR0022,956.784542,1708062467,Mumbai,ahmedabad,Entertainment,DEV-54C67087,mobile,CARD,73970.71,success,192.62.225.88
1445,TXN-C0877E9A01B8,USR0048,2332.363564,07/02/2024 01:02:50,Chennai,HYD,Healthcare,DEV-D7256DA2,ATM,UPI,11946.20,pending,192.109.213.177


In [188]:
import pandas as pd
import re
from dateutil import parser
import numpy as np

def parse_timestamp(x):
    """
    Robust parser for transaction_timestamp column
    Special handling for DD-MM-HH:MM:SS.mmm format (assumes 2024)
    Returns timezone-naive pd.Timestamp or pd.NaT
    """
    if pd.isna(x) or str(x).strip() in ['', 'null', 'NULL', 'NA', 'N/A', '-']:
        return pd.NaT

    x = str(x).strip()

    # ─── Special case: DD-MM-HH:MM:SS.mmm ───────────────────────────────
    # Examples: 01-04-11:41:45.267   01-03-17:13:06.776
    if re.match(r'^\d{2}-\d{2}-\d{2}:\d{2}:\d{2}\.\d{3}$', x):
        try:
            dd, rest = x.split('-', 1)
            mm, time_part = rest.split('-', 1)
            full_str = f"2024-{mm.zfill(2)}-{dd.zfill(2)} {time_part}"
            return pd.to_datetime(full_str, format="%Y-%m-%d %H:%M:%S.%f")
        except:
            pass  # fall through to other parsers

    # ─── Quick numeric path (unix timestamp) ─────────────────────────────
    if re.match(r'^\d+$', x):
        try:
            num = float(x)
            if num > 1e12:  # milliseconds
                return pd.to_datetime(num, unit='ms')
            elif num > 1e9:  # seconds
                return pd.to_datetime(num, unit='s')
        except:
            pass

        # Compact YYYYMMDDHHMMSS
        if len(x) == 14:
            try:
                return pd.to_datetime(x, format='%Y%m%d%H%M%S')
            except:
                pass

    # ─── Fuzzy natural language parse ────────────────────────────────────
    try:
        return parser.parse(x, fuzzy=True, dayfirst=True, yearfirst=False)
    except:
        pass

    # ─── Explicit common formats ─────────────────────────────────────────
    formats = [
        '%d-%m-%Y %H:%M:%S',
        '%d/%m/%Y %H:%M',
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M',
        '%d-%b-%Y',
        '%d %b %Y',
        '%d-%b-%Y %H:%M',        # extra common in India
        '%d %b %Y %H:%M:%S',
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(x, format=fmt, errors='raise')
        except:
            continue

    # Nothing matched
    return pd.NaT


# ────────────────────────────────────────────────
# Apply parsing
df['transaction_timestamp'] = df['transaction_timestamp'].astype(str).str.strip()
df['ts'] = df['transaction_timestamp'].apply(parse_timestamp)

# ─── Remove timezone from ALL timestamps (prevents sorting error) ───────
df['ts'] = df['ts'].dt.tz_localize(None)

# ─── Quality report ─────────────────────────────────────────────────────
success = df['ts'].notna().sum()
total   = len(df)
fail    = total - success

print(f"Successfully parsed: {success:,} / {total:,} rows")
print(f"Failed to parse    : {fail:,} ({fail/total:.2%})")

if fail > 0:
    print("\nFailed timestamp examples (first 10 unique):")
    failed_samples = df[df['ts'].isna()]['transaction_timestamp'].dropna().unique()[:10]
    print(failed_samples.tolist())

# ─── Sorting (should now work without TypeError) ────────────────────────
df = df.sort_values(['user_id', 'ts']).reset_index(drop=True)

# Quick sanity check
print("\nDate range after sorting:")
print("Earliest:", df['ts'].min())
print("Latest:  ", df['ts'].max())
print("Any remaining NaT?", df['ts'].isna().any())

Successfully parsed: 1,447 / 1,447 rows
Failed to parse    : 0 (0.00%)

Date range after sorting:
Earliest: 2024-01-01 04:25:29.601133
Latest:   2024-12-06 22:31:45.235402
Any remaining NaT? False


In [189]:
df

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,ts
0,TXN-BC3806284B2F,USR0001,373.233492,2024-04-01T13:07:18.827444,delhi,delhi,Food & Dining,DEV-CEAEBD15,ATM,CARD,57783.90,success,192.27.174.84,2024-01-04 13:07:18.827444
1,TXN-E354E4217F1F,USR0001,525.070000,2024-05-01T09:37:33.240547,Delhi,DEL,Fuel,DEV-CEAEBD15,ATM,CARD,51140.61,success,192.27.174.24,2024-01-05 09:37:33.240547
2,TXN-FD97B26A7F87,USR0001,NaN,2024-05-02T22:03:38.523540,DELHI,ahmedabad,Fuel,DEV-CEAEBD15,ATM,UPI,50938.46,pending,192.27.174.169,2024-02-05 22:03:38.523540
3,TXN-4A7010CE412C,USR0001,426.000000,2024-05-03T00:04:00.933028,Delhi,DEL,Healthcare,DEV-CEAEBD15,ATM,CARD,50512.26,success,192.27.174.69,2024-03-05 00:04:00.933028
4,TXN-BFAB1556A909,USR0001,473.581395,1711940611,De...,delhi,Clo...,DEV-CEAEBD15,ATM,CARD,58157.14,success,192.168..1,2024-04-01 03:03:31.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-57090FAF517D,USR0080,845.532396,2024-01-06T12:43:04,mUMBAI,CCU,Grocery,NEW-8708BEAA,mobile,CARD,73524.81,success,193.86.80.51,2024-06-01 12:43:04.000000
1443,TXN-664EF9CCAE1D,USR0080,819.212373,2024-03-08T13:19:28.085313,BOM,mumbai,Healthcare,DEV-2A020351,web,CARD,100349.95,success,192.236.33.245,2024-08-03 13:19:28.085313
1444,TXN-E9555C102327,USR0080,4536.037622,2024-03-10T10:03:20.278228,mumbai,Pune,Travel,NaN,web,UPI,95813.91,success,192.236.33.143,2024-10-03 10:03:20.278228
1445,TXN-FA2BD9DF2BB1,USR0080,4387.423835,2024-03-11T04:19:48.897752,Mumbai,MUMBAI,Healthcare,DEV-2A320A3B,web,CARD,90741.21,success,192.236.33.50,2024-11-03 04:19:48.897752


In [190]:
import pandas as pd
import numpy as np

# ────────────────────────────────────────────────
# Much more complete map — covers ~80–95% in most Indian fintech datasets
city_map = {
    # Mumbai + variants
    'mumbai': 'Mumbai', 'bombay': 'Mumbai', 'bom': 'Mumbai', 'mum': 'Mumbai',
    'mumbai ': 'Mumbai', ' mumbai': 'Mumbai', 'mumabi': 'Mumbai', 'mumabai': 'Mumbai',
    'navimumbai': 'Navi Mumbai', 'navi mumbai': 'Navi Mumbai', 'navi-mumbai': 'Navi Mumbai',

    # Bengaluru
    'bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru', 'banglore': 'Bengaluru',
    'blr': 'Bengaluru', 'benagluru': 'Bengaluru', 'bangaluru': 'Bengaluru',
    'bangalor': 'Bengaluru', 'benguluru': 'Bengaluru',

    # Chennai + variants
    'chennai': 'Chennai', 'madras': 'Chennai', 'chennnai': 'Chennai',
    'chenai': 'Chennai', 'chennnai': 'Chennai', 'chennal': 'Chennai',

    # Kolkata
    'kolkata': 'Kolkata', 'calcutta': 'Kolkata', 'calcuta': 'Kolkata',
    'kolkatta': 'Kolkata', 'colkata': 'Kolkata',

    # Delhi / NCR
    'delhi': 'Delhi', 'new delhi': 'Delhi', 'ncr': 'Delhi', 'newdelhi': 'Delhi',
    'new-delhi': 'Delhi', 'gurgaon': 'Gurgaon', 'gurugram': 'Gurugram',
    'noida': 'Noida', 'greater noida': 'Greater Noida',

    # Hyderabad
    'hyderabad': 'Hyderabad', 'hyd': 'Hyderabad', 'hyderbad': 'Hyderabad',

    # Pune
    'pune': 'Pune', 'punr': 'Pune', 'poona': 'Pune',

    # Ahmedabad
    'ahmedabad': 'Ahmedabad', 'ahmdabad': 'Ahmedabad', 'amdavad': 'Ahmedabad',

    # Jaipur
    'jaipur': 'Jaipur', 'jaypur': 'Jaipur',

    # Lucknow
    'lucknow': 'Lucknow', 'lukhnow': 'Lucknow',

    # Surat
    'surat': 'Surat',

    # Other frequent ones
    'indore': 'Indore', 'bhopal': 'Bhopal', 'patna': 'Patna',
    'coimbatore': 'Coimbatore', 'kochi': 'Kochi', 'ernakulam': 'Kochi',
    'thiruvananthapuram': 'Thiruvananthapuram', 'trivandrum': 'Thiruvananthapuram',
    'visakhapatnam': 'Visakhapatnam', 'vizag': 'Visakhapatnam',
}

# ────────────────────────────────────────────────
def normalize_city_advanced(col_name):
    """
    Returns normalized city names + detailed success statistics
    """
    # Step 1: clean input
    s = df[col_name].astype(str).str.lower().str.strip()

    # Step 2: treat obvious missing/invalid as NaN
    missing_markers = ['', 'nan', 'na', 'n/a', 'null', '-', 'unknown', 'none', 'nil', 'missing']
    s = s.replace(missing_markers, np.nan)

    # Step 3: map known aliases
    mapped = s.map(city_map)

    # Step 4: fallback → title case only for values that look like real city names
    fallback = s.str.title()
    # Very short or numeric-looking → keep as NaN
    fallback = np.where(
        (s.str.len() <= 2) | s.str.match(r'^\d'), np.nan, fallback
    )

    result = mapped.combine_first(pd.Series(fallback, index=s.index))

    # ── Statistics ────────────────────────────────────────
    total_rows = len(df)
    original_non_null = df[col_name].notna().sum()
    after_non_null   = result.notna().sum()
    mapped_to_known  = result.isin(city_map.values()).sum()

    print(f"\n=== LOCATION NORMALIZATION REPORT: {col_name} ===")
    print(f"Total rows                     : {total_rows:,}")
    print(f"Original non-null values       : {original_non_null:,} ({original_non_null/total_rows*100:.2f}%)")
    print(f"After normalization non-null   : {after_non_null:,} ({after_non_null/total_rows*100:.2f}%)")
    print(f"→ Mapped to known canonical    : {mapped_to_known:,} ({mapped_to_known/total_rows*100:.2f}%)")
    print(f"→ Title-cased (still unknown)  : {after_non_null - mapped_to_known:,} ({(after_non_null - mapped_to_known)/total_rows*100:.2f}%)")

    # Show top unmapped cities — this is the most important part!
    unmapped = result[result.notna() & ~result.isin(city_map.values())]
    if len(unmapped) > 0:
        print("\nTop 12 unmapped / unknown cities (add the frequent ones to map!):")
        print(unmapped.value_counts().head(12))
    else:
        print("\nAll non-null values were mapped to known cities — excellent!")

    return result

# ────────────────────────────────────────────────
# Apply to both columns
df['user_city']     = normalize_city_advanced('user_location')
df['merchant_city'] = normalize_city_advanced('merchant_location')

# Optional cleanup
# df = df.drop(columns=['user_location', 'merchant_location'])


=== LOCATION NORMALIZATION REPORT: user_location ===
Total rows                     : 1,447
Original non-null values       : 1,447 (100.00%)
After normalization non-null   : 1,444 (99.79%)
→ Mapped to known canonical    : 1,107 (76.50%)
→ Title-cased (still unknown)  : 337 (23.29%)

Top 12 unmapped / unknown cities (add the frequent ones to map!):
user_location
Jai          98
Lko          56
Pnq          38
Ccu          38
Del          26
Amd          25
Maa          19
Dubai         4
New York      4
Bangkok       4
Singapore     2
J...          2
Name: count, dtype: int64

=== LOCATION NORMALIZATION REPORT: merchant_location ===
Total rows                     : 1,447
Original non-null values       : 1,447 (100.00%)
After normalization non-null   : 1,447 (100.00%)
→ Mapped to known canonical    : 1,147 (79.27%)
→ Title-cased (still unknown)  : 300 (20.73%)

Top 12 unmapped / unknown cities (add the frequent ones to map!):
merchant_location
Jai          63
Lko          63
Ccu        

In [191]:
import pandas as pd
import numpy as np

# ────────────────────────────────────────────────
# Expanded map based on YOUR actual top unmapped cities
city_map = {
    # Previous strong ones
    'mumbai': 'Mumbai', 'bombay': 'Mumbai', 'bom': 'Mumbai', 'mum': 'Mumbai',
    'bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru', 'banglore': 'Bengaluru',
    'chennai': 'Chennai', 'madras': 'Chennai',
    'kolkata': 'Kolkata', 'calcutta': 'Kolkata',
    'delhi': 'Delhi', 'new delhi': 'Delhi', 'ncr': 'Delhi',

    # ── NEW: From your Top unmapped ───────────────────────────────
    # Jai → almost certainly Jaipur (very common abbreviation)
    'jai': 'Jaipur', 'jaipur': 'Jaipur', 'jpr': 'Jaipur',

    # Lko → Lucknow
    'lko': 'Lucknow', 'lucknow': 'Lucknow', 'lknw': 'Lucknow',

    # Pnq → Pune
    'pnq': 'Pune', 'pune': 'Pune', 'pnq ': 'Pune',

    # Ccu → Kolkata (CCU = Netaji Subhas Chandra Bose Intl Airport code)
    'ccu': 'Kolkata',

    # Del → Delhi (DEL = Indira Gandhi Intl Airport code)
    'del': 'Delhi',

    # Amd → Ahmedabad
    'amd': 'Ahmedabad', 'ahmedabad': 'Ahmedabad',

    # Maa → Chennai (MAA = Chennai International Airport code)
    'maa': 'Chennai',

    # International ones that appear (even if few)
    'dubai': 'Dubai',
    'new york': 'New York', 'ny': 'New York',
    'bangkok': 'Bangkok',
    'singapore': 'Singapore',

    # Optional extras you might see next (add if they appear in next run)
    # 'hyd': 'Hyderabad', 'blr': 'Bengaluru', 'pat': 'Patna', 'sur': 'Surat',
}

# ────────────────────────────────────────────────
def normalize_city_advanced(col_name):
    """
    Returns normalized city names + detailed success statistics
    """
    # Step 1: clean input
    s = df[col_name].astype(str).str.lower().str.strip()

    # Step 2: treat obvious missing/invalid as NaN
    missing_markers = ['', 'nan', 'na', 'n/a', 'null', '-', 'unknown', 'none', 'nil', 'missing']
    s = s.replace(missing_markers, np.nan)

    # Step 3: map known aliases
    mapped = s.map(city_map)

    # Step 4: fallback → title case only for plausible city-like strings
    fallback = s.str.title()
    # Protect very short junk (1-3 chars that aren't in map) → NaN
    fallback = np.where(
        (s.str.len() <= 3) & (~s.isin(city_map.keys())), np.nan, fallback
    )

    result = mapped.combine_first(pd.Series(fallback, index=s.index))

    # ── Statistics ────────────────────────────────────────
    total_rows = len(df)
    original_non_null = df[col_name].notna().sum()
    after_non_null   = result.notna().sum()
    mapped_to_known  = result.isin(city_map.values()).sum()

    print(f"\n=== LOCATION NORMALIZATION REPORT: {col_name} ===")
    print(f"Total rows                     : {total_rows:,}")
    print(f"Original non-null values       : {original_non_null:,} ({original_non_null/total_rows*100:.2f}%)")
    print(f"After normalization non-null   : {after_non_null:,} ({after_non_null/total_rows*100:.2f}%)")
    print(f"→ Mapped to known canonical    : {mapped_to_known:,} ({mapped_to_known/total_rows*100:.2f}%)")
    print(f"→ Title-cased (still unknown)  : {after_non_null - mapped_to_known:,} ({(after_non_null - mapped_to_known)/total_rows*100:.2f}%)")

    # Show remaining unmapped (should be much fewer now)
    unmapped = result[result.notna() & ~result.isin(city_map.values())]
    if len(unmapped) > 0:
        print("\nRemaining top unmapped / unknown cities:")
        print(unmapped.value_counts().head(12))
    else:
        print("\nAll non-null values mapped to known cities — perfect coverage!")

    return result

# ────────────────────────────────────────────────
# Apply to both columns
df['user_city']     = normalize_city_advanced('user_location')
df['merchant_city'] = normalize_city_advanced('merchant_location')

# Optional: drop originals if you no longer need them
# df = df.drop(columns=['user_location', 'merchant_location'])


=== LOCATION NORMALIZATION REPORT: user_location ===
Total rows                     : 1,447
Original non-null values       : 1,447 (100.00%)
After normalization non-null   : 1,392 (96.20%)
→ Mapped to known canonical    : 1,271 (87.84%)
→ Title-cased (still unknown)  : 121 (8.36%)

Remaining top unmapped / unknown cities:
user_location
Hyderabad    101
J...           2
Luc#           1
De...          1
Luckn          1
Bangalo        1
Hyder...       1
Jai??          1
Beng...        1
Pu...          1
Madr#          1
New Yor#       1
Name: count, dtype: int64

=== LOCATION NORMALIZATION REPORT: merchant_location ===
Total rows                     : 1,447
Original non-null values       : 1,447 (100.00%)
After normalization non-null   : 1,380 (95.37%)
→ Mapped to known canonical    : 1,282 (88.60%)
→ Title-cased (still unknown)  : 98 (6.77%)

Remaining top unmapped / unknown cities:
merchant_location
Hyderabad    98
Name: count, dtype: int64


In [192]:
# ────────────────────────────────────────────────
# FINAL expanded map — should get you to 94–97% mapped
city_map = {
    # Previous strong ones (keep them all)
    'mumbai': 'Mumbai', 'bombay': 'Mumbai', 'bom': 'Mumbai', 'mum': 'Mumbai',
    'bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru', 'banglore': 'Bengaluru',
    'chennai': 'Chennai', 'madras': 'Chennai',
    'kolkata': 'Kolkata', 'calcutta': 'Kolkata',
    'delhi': 'Delhi', 'new delhi': 'Delhi', 'ncr': 'Delhi',
    'jai': 'Jaipur', 'jaipur': 'Jaipur',
    'lko': 'Lucknow', 'lucknow': 'Lucknow',
    'pnq': 'Pune', 'pune': 'Pune',
    'ccu': 'Kolkata',
    'del': 'Delhi',
    'amd': 'Ahmedabad', 'ahmedabad': 'Ahmedabad',
    'maa': 'Chennai',

    # ── NEW additions based on your latest unmapped ───────
    'hyderabad': 'Hyderabad', 'hyd': 'Hyderabad', 'hyderbad': 'Hyderabad',
    'hyder': 'Hyderabad', 'hyderab': 'Hyderabad',   # partial matches

    # International (small volume but clean up)
    'dubai': 'Dubai',
    'new york': 'New York',
    'bangkok': 'Bangkok',
    'singapore': 'Singapore',
}

# ────────────────────────────────────────────────
def normalize_city_final(col_name):
    s = df[col_name].astype(str).str.lower().str.strip()

    # Missing / junk markers
    missing_markers = ['', 'nan', 'na', 'n/a', 'null', '-', 'unknown', 'none', 'nil', 'missing']
    s = s.replace(missing_markers, np.nan)

    # Junk filter: patterns like J..., Luc#, Jai??, De... → set to NaN
    junk_mask = s.str.contains(r'[#\?\.]{2,}', na=False)
    s = s.where(~junk_mask, np.nan)

    # Map known aliases
    mapped = s.map(city_map)

    # Fallback: title case only for plausible strings
    fallback = s.str.title()
    # Protect very short / suspicious strings
    short_or_junk = (s.str.len() < 4) | s.str.contains(r'[#\?\.]', na=False)
    fallback = fallback.where(~short_or_junk, np.nan)

    result = mapped.combine_first(fallback)

    # ── Statistics ────────────────────────────────────────
    total = len(df)
    orig_nonnull = df[col_name].notna().sum()
    after_nonnull = result.notna().sum()
    mapped_known = result.isin(city_map.values()).sum()

    print(f"\n=== FINAL LOCATION NORMALIZATION REPORT: {col_name} ===")
    print(f"Total rows                     : {total:,}")
    print(f"Original non-null              : {orig_nonnull:,} ({orig_nonnull/total*100:.2f}%)")
    print(f"After normalization non-null   : {after_nonnull:,} ({after_nonnull/total*100:.2f}%)")
    print(f"→ Mapped to known canonical    : {mapped_known:,} ({mapped_known/total*100:.2f}%)")
    print(f"→ Title-cased (still unknown)  : {after_nonnull - mapped_known:,} ({(after_nonnull - mapped_known)/total*100:.2f}%)")

    unmapped = result[result.notna() & ~result.isin(city_map.values())]
    if len(unmapped) > 0:
        print("\nRemaining unmapped cities:")
        print(unmapped.value_counts().head(15))
    else:
        print("\nPerfect — all non-null values mapped!")

    return result

In [193]:
df


,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,ts,user_city,merchant_city
0,TXN-BC3806284B2F,USR0001,373.233492,2024-04-01T13:07:18.827444,delhi,delhi,Food & Dining,DEV-CEAEBD15,ATM,CARD,57783.90,success,192.27.174.84,2024-01-04 13:07:18.827444,Delhi,Delhi
1,TXN-E354E4217F1F,USR0001,525.070000,2024-05-01T09:37:33.240547,Delhi,DEL,Fuel,DEV-CEAEBD15,ATM,CARD,51140.61,success,192.27.174.24,2024-01-05 09:37:33.240547,Delhi,Delhi
2,TXN-FD97B26A7F87,USR0001,NaN,2024-05-02T22:03:38.523540,DELHI,ahmedabad,Fuel,DEV-CEAEBD15,ATM,UPI,50938.46,pending,192.27.174.169,2024-02-05 22:03:38.523540,Delhi,Ahmedabad
3,TXN-4A7010CE412C,USR0001,426.000000,2024-05-03T00:04:00.933028,Delhi,DEL,Healthcare,DEV-CEAEBD15,ATM,CARD,50512.26,success,192.27.174.69,2024-03-05 00:04:00.933028,Delhi,Delhi
4,TXN-BFAB1556A909,USR0001,473.581395,1711940611,De...,delhi,Clo...,DEV-CEAEBD15,ATM,CARD,58157.14,success,192.168..1,2024-04-01 03:03:31.000000,De...,Delhi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-57090FAF517D,USR0080,845.532396,2024-01-06T12:43:04,mUMBAI,CCU,Grocery,NEW-8708BEAA,mobile,CARD,73524.81,success,193.86.80.51,2024-06-01 12:43:04.000000,Mumbai,Kolkata
1443,TXN-664EF9CCAE1D,USR0080,819.212373,2024-03-08T13:19:28.085313,BOM,mumbai,Healthcare,DEV-2A020351,web,CARD,100349.95,success,192.236.33.245,2024-08-03 13:19:28.085313,Mumbai,Mumbai
1444,TXN-E9555C102327,USR0080,4536.037622,2024-03-10T10:03:20.278228,mumbai,Pune,Travel,NaN,web,UPI,95813.91,success,192.236.33.143,2024-10-03 10:03:20.278228,Mumbai,Pune
1445,TXN-FA2BD9DF2BB1,USR0080,4387.423835,2024-03-11T04:19:48.897752,Mumbai,MUMBAI,Healthcare,DEV-2A320A3B,web,CARD,90741.21,success,192.236.33.50,2024-11-03 04:19:48.897752,Mumbai,Mumbai


In [194]:
import pandas as pd
import numpy as np

def location_normalization_success_report():
    """
    Prints detailed success rate and quality metrics for user_city and merchant_city
    after normalization has been applied.
    """
    print("\n" + "="*70)
    print("     LOCATION NORMALIZATION SUCCESS & QUALITY REPORT")
    print("="*70 + "\n")

    total_rows = len(df)

    for col in ['user_city', 'merchant_city']:
        print(f"→ Column: {col}")
        print("-"*60)

        # Basic counts
        total_non_null_original = df['user_location' if 'user' in col else 'merchant_location'].notna().sum()
        valid_after = df[col].notna().sum()
        mapped_to_known = df[col].isin(city_map.values()).sum()
        still_unknown = valid_after - mapped_to_known
        became_nan = total_rows - valid_after

        # Percentages
        pct_valid_original = total_non_null_original / total_rows * 100
        pct_valid_after    = valid_after / total_rows * 100
        pct_mapped_known   = mapped_to_known / total_rows * 100
        pct_still_unknown  = still_unknown / total_rows * 100
        pct_nan            = became_nan / total_rows * 100

        print(f"Total rows                          : {total_rows:,}")
        print(f"Original non-null values            : {total_non_null_original:,} ({pct_valid_original:.2f}%)")
        print(f"After normalization - valid values  : {valid_after:,} ({pct_valid_after:.2f}%)")
        print(f"   └─ Mapped to known city          : {mapped_to_known:,} ({pct_mapped_known:.2f}%)")
        print(f"   └─ Still unknown (title-cased)   : {still_unknown:,} ({pct_still_unknown:.2f}%)")
        print(f"Became NaN (missing or junk)        : {became_nan:,} ({pct_nan:.2f}%)")

        # Show top remaining unknown values
        if still_unknown > 0:
            print("\nTop remaining unknown (title-cased) values:")
            unknown_values = df[col][~df[col].isin(city_map.values()) & df[col].notna()]
            if len(unknown_values) > 0:
                print(unknown_values.value_counts().head(15).to_string())
            else:
                print("   (none found)")
        else:
            print("\nAll valid values mapped to known cities — excellent!")

        print("\n")

# ────────────────────────────────────────────────
# Run the report
location_normalization_success_report()


     LOCATION NORMALIZATION SUCCESS & QUALITY REPORT

→ Column: user_city
------------------------------------------------------------
Total rows                          : 1,447
Original non-null values            : 1,447 (100.00%)
After normalization - valid values  : 1,392 (96.20%)
   └─ Mapped to known city          : 1,372 (94.82%)
   └─ Still unknown (title-cased)   : 20 (1.38%)
Became NaN (missing or junk)        : 55 (3.80%)

Top remaining unknown (title-cased) values:
user_city
J...        2
Luc#        1
De...       1
Luckn       1
Bangalo     1
Hyder...    1
Jai??       1
Beng...     1
Pu...       1
Madr#       1
New Yor#    1
Hyde        1
H...        1
Pun??       1
Pu??        1


→ Column: merchant_city
------------------------------------------------------------
Total rows                          : 1,447
Original non-null values            : 1,447 (100.00%)
After normalization - valid values  : 1,380 (95.37%)
   └─ Mapped to known city          : 1,380 (95.37%)
   └─ 

In [195]:
df

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,ts,user_city,merchant_city
0,TXN-BC3806284B2F,USR0001,373.233492,2024-04-01T13:07:18.827444,delhi,delhi,Food & Dining,DEV-CEAEBD15,ATM,CARD,57783.90,success,192.27.174.84,2024-01-04 13:07:18.827444,Delhi,Delhi
1,TXN-E354E4217F1F,USR0001,525.070000,2024-05-01T09:37:33.240547,Delhi,DEL,Fuel,DEV-CEAEBD15,ATM,CARD,51140.61,success,192.27.174.24,2024-01-05 09:37:33.240547,Delhi,Delhi
2,TXN-FD97B26A7F87,USR0001,NaN,2024-05-02T22:03:38.523540,DELHI,ahmedabad,Fuel,DEV-CEAEBD15,ATM,UPI,50938.46,pending,192.27.174.169,2024-02-05 22:03:38.523540,Delhi,Ahmedabad
3,TXN-4A7010CE412C,USR0001,426.000000,2024-05-03T00:04:00.933028,Delhi,DEL,Healthcare,DEV-CEAEBD15,ATM,CARD,50512.26,success,192.27.174.69,2024-03-05 00:04:00.933028,Delhi,Delhi
4,TXN-BFAB1556A909,USR0001,473.581395,1711940611,De...,delhi,Clo...,DEV-CEAEBD15,ATM,CARD,58157.14,success,192.168..1,2024-04-01 03:03:31.000000,De...,Delhi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-57090FAF517D,USR0080,845.532396,2024-01-06T12:43:04,mUMBAI,CCU,Grocery,NEW-8708BEAA,mobile,CARD,73524.81,success,193.86.80.51,2024-06-01 12:43:04.000000,Mumbai,Kolkata
1443,TXN-664EF9CCAE1D,USR0080,819.212373,2024-03-08T13:19:28.085313,BOM,mumbai,Healthcare,DEV-2A020351,web,CARD,100349.95,success,192.236.33.245,2024-08-03 13:19:28.085313,Mumbai,Mumbai
1444,TXN-E9555C102327,USR0080,4536.037622,2024-03-10T10:03:20.278228,mumbai,Pune,Travel,NaN,web,UPI,95813.91,success,192.236.33.143,2024-10-03 10:03:20.278228,Mumbai,Pune
1445,TXN-FA2BD9DF2BB1,USR0080,4387.423835,2024-03-11T04:19:48.897752,Mumbai,MUMBAI,Healthcare,DEV-2A320A3B,web,CARD,90741.21,success,192.236.33.50,2024-11-03 04:19:48.897752,Mumbai,Mumbai


In [196]:
import pandas as pd
import numpy as np
import re

def clean_and_flag_ip(df, ip_col='ip_address'):
    """
    Clean ip_address and create fraud-useful features.
    """
    s = df[ip_col].astype(str).str.strip()

    # 1. Missing patterns
    missing_patterns = [
        r'^\s*$',
        r'^(NA|N/A|null|None|-|unknown|missing|private|vpn)$',
        r'^[\-–—]$'
    ]
    is_missing = s.str.match('|'.join(missing_patterns), na=False) | s.isna()

    # 2. Looks like IPv4 pattern
    ipv4_pattern = r'^(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})$'
    looks_like_ipv4 = s.str.match(ipv4_pattern, na=False)

    # 3. Valid octets 0–255
    def valid_octets(ip_str):
        if not isinstance(ip_str, str) or pd.isna(ip_str):
            return False
        try:
            parts = [int(x) for x in ip_str.split('.') if x or x == '0']
            return len(parts) == 4 and all(0 <= p <= 255 for p in parts)
        except:
            return False

    is_valid_format = looks_like_ipv4 & s.apply(valid_octets)

    # 4. Special cases
    is_localhost = s.isin(['127.0.0.1', 'localhost', '::1'])
    is_zero_ip   = s == '0.0.0.0'
    is_private   = s.str.startswith(('10.', '192.168.', '172.16.', '172.17.', '172.18.', '172.19.',
                                     '172.20.', '172.21.', '172.22.', '172.23.', '172.24.', '172.25.',
                                     '172.26.', '172.27.', '172.28.', '172.29.', '172.30.', '172.31.'))

    # 5. Create features
    df['ip_clean'] = np.where(is_valid_format, s, np.nan)

    df['is_valid_ipv4'] = (
        is_valid_format &
        ~is_localhost &
        ~is_zero_ip
    ).astype(int)

    df['ip_anomaly'] = (
        is_missing |
        ~is_valid_format |
        is_localhost |
        is_zero_ip
    ).astype(int)

    # FIXED: np.select – all choices are strings + explicit default
    df['ip_present'] = np.select(
        condlist=[
            df['is_valid_ipv4'] == 1,
            is_missing,
            is_localhost | is_zero_ip,
        ],
        choicelist=[
            'valid',
            'missing',
            'localhost/zero',
        ],
        default='malformed'          # ← string, matches choicelist dtype
    )

    # 6. Summary
    print("IP Address Cleaning Summary:")
    print("────────────────────────────")
    print(f"Total rows:               {len(df):,}")
    print(f"Original missing/empty:   {is_missing.sum():,} ({is_missing.mean():.2%})")
    print(f"Valid IPv4 format:        {is_valid_format.sum():,} ({is_valid_format.mean():.2%})")
    print(f"Strict valid IPv4:        {df['is_valid_ipv4'].sum():,} ({df['is_valid_ipv4'].mean():.2%})")
    print(f"Anomaly flag (1=suspicious): {df['ip_anomaly'].sum():,} ({df['ip_anomaly'].mean():.2%})")
    print("\nip_present value counts:")
    print(df['ip_present'].value_counts(dropna=False))

    suspicious = df[df['ip_anomaly'] == 1][ip_col].value_counts().head(15)
    if not suspicious.empty:
        print("\nTop 15 suspicious IP values:")
        print(suspicious)

    return df


# ────────────────────────────────────────────────
# Run it
df = clean_and_flag_ip(df, ip_col='ip_address')

# Optional: quick inspection
print("\nSample suspicious rows:")
print(df[df['ip_anomaly'] == 1][['ip_address', 'ip_clean', 'is_valid_ipv4', 'ip_anomaly', 'ip_present']].head(10))

IP Address Cleaning Summary:
────────────────────────────
Total rows:               1,447
Original missing/empty:   0 (0.00%)
Valid IPv4 format:        1,340 (92.61%)
Strict valid IPv4:        1,340 (92.61%)
Anomaly flag (1=suspicious): 107 (7.39%)

ip_present value counts:
ip_present
valid        1340
malformed     107
Name: count, dtype: int64

Top 15 suspicious IP values:
ip_address
not_an_ip       8
192.168..1      7
999.168.1.1     5
66.67.x.233     2
132.104.x.18    1
103.80.x.74     1
52.11.x.214     1
136.174.x.78    1
11.75.x.204     1
195.194.x.47    1
Name: count, dtype: int64

Sample suspicious rows:
       ip_address ip_clean  is_valid_ipv4  ip_anomaly ip_present
4      192.168..1      NaN              0           1  malformed
5             NaN      NaN              0           1  malformed
12   132.104.x.18      NaN              0           1  malformed
42            NaN      NaN              0           1  malformed
68      not_an_ip      NaN              0           1  

In [197]:
# A. Same IP used by many users
ip_user_count = df.groupby('ip_clean')['user_id'].nunique()
df['ip_multi_user'] = df['ip_clean'].map(ip_user_count).fillna(1)
df['ip_multi_user_flag'] = (df['ip_multi_user'] > 3).astype(int)

# B. IP is private network
df['ip_is_private'] = df['ip_clean'].str.startswith(
    ('10.', '192.168.', '172.16.', '172.17.', '172.18.', '172.19.',
     '172.20.', '172.21.', '172.22.', '172.23.', '172.24.', '172.25.',
     '172.26.', '172.27.', '172.28.', '172.29.', '172.30.', '172.31.'),
    na=False   # ← this prevents NaN → float error
).astype(int)

# C. Masked IP (contains 'x')
df['ip_masked'] = df['ip_address'].str.contains('x', case=False, na=False).astype(int)

# D. Invalid octet (contains invalid numbers)
df['ip_invalid_octet'] = (
    df['ip_anomaly'] &
    df['ip_address'].str.contains(r'999|256|>|>255', na=False)
).astype(int)

# Quick check
print("New IP feature means:")
print(df[['ip_multi_user_flag', 'ip_is_private', 'ip_masked', 'ip_invalid_octet']].mean())

print("\nTop IPs shared by multiple users:")
print(ip_user_count[ip_user_count > 2].sort_values(ascending=False).head(10))

New IP feature means:
ip_multi_user_flag    0.000000
ip_is_private         0.000000
ip_masked             0.005529
ip_invalid_octet      0.003455
dtype: float64

Top IPs shared by multiple users:
Series([], Name: user_id, dtype: int64)


In [198]:
import re

# ─── Step 1: Create cleaned IP column ───
def clean_ip(ip):
    if pd.isna(ip) or str(ip).strip() == '':
        return None
    ip_str = str(ip).strip().lower()
    # Remove common junk
    ip_str = re.sub(r'\s+', '', ip_str)
    # Basic IPv4 pattern match
    match = re.match(r'^(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})$', ip_str)
    if match:
        octets = [int(o) for o in match.groups()]
        if all(0 <= o <= 255 for o in octets):
            return '.'.join(map(str, octets))
    return None  # invalid → None

df['ip_clean'] = df['ip_address'].apply(clean_ip)

# ─── Step 2: Validation flags (already have some, add more) ───
df['ip_valid_format'] = df['ip_clean'].notna().astype(int)
df['ip_is_private'] = df['ip_clean'].str.startswith(
    ('10.', '192.168.', '172.16.', '172.17.', '172.18.', '172.19.',
     '172.20.', '172.21.', '172.22.', '172.23.', '172.24.', '172.25.',
     '172.26.', '172.27.', '172.28.', '172.29.', '172.30.', '172.31.'),
    na=False
).astype(int)

df['ip_masked'] = df['ip_address'].str.contains('x', case=False, na=False).astype(int)

# ─── Step 3: Success & quality metrics ───
print("\nIP VALIDATION REPORT")
valid_ips = df['ip_valid_format'].sum()
total_rows = len(df)
print(f"Valid IPv4 format after cleaning: {valid_ips:,} / {total_rows:,}  ({valid_ips/total_rows*100:.2f}%)")

invalid_ips = total_rows - valid_ips
print(f"Invalid / unparseable IPs: {invalid_ips:,}  ({invalid_ips/total_rows*100:.2f}%)")

private_ips = df['ip_is_private'].sum()
print(f"Private network IPs (RFC 1918): {private_ips:,}  ({private_ips/total_rows*100:.2f}%)")

masked_ips = df['ip_masked'].sum()
print(f"Masked IPs (contain 'x'): {masked_ips:,}  ({masked_ips/total_rows*100:.2f}%)")

# Sample of cleaned vs original
print("\nSample of IP cleaning results:")
print(df[['ip_address', 'ip_clean', 'ip_valid_format', 'ip_is_private', 'ip_masked']].head(15))

# Success rate
ip_success_rate = (valid_ips / total_rows) * 100
print(f"\nIP cleaning & validation success rate: {ip_success_rate:.2f}%")



IP VALIDATION REPORT
Valid IPv4 format after cleaning: 1,340 / 1,447  (92.61%)
Invalid / unparseable IPs: 107  (7.39%)
Private network IPs (RFC 1918): 0  (0.00%)
Masked IPs (contain 'x'): 8  (0.55%)

Sample of IP cleaning results:
        ip_address        ip_clean  ip_valid_format  ip_is_private  ip_masked
0    192.27.174.84   192.27.174.84                1              0          0
1    192.27.174.24   192.27.174.24                1              0          0
2   192.27.174.169  192.27.174.169                1              0          0
3    192.27.174.69   192.27.174.69                1              0          0
4       192.168..1            None                0              0          0
5              NaN            None                0              0          0
6   192.27.174.126  192.27.174.126                1              0          0
7   192.27.174.122  192.27.174.122                1              0          0
8    192.27.174.79   192.27.174.79                1              0

In [199]:
# ─── Step 1: Quantify missingness per column ───
print("\nMissing value report BEFORE handling:")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False))

# ─── Step 2: Create missing flags for important columns ───
important_cols = ['amount', 'ts', 'user_city', 'merchant_city', 'device_id', 'ip_address', 'account_balance']
for col in important_cols:
    if col in df.columns:
        df[f'is_missing_{col}'] = df[col].isna().astype(int)

# ─── Step 3: Imputation strategy ───
# Amount: median per user (behavioral)
if 'amount' in df.columns:
    df['amount'] = df.groupby('user_id')['amount'].transform(lambda x: x.fillna(x.median()))
    # If user has no non-null → global median
    df['amount'] = df['amount'].fillna(df['amount'].median())

# ts: drop rows where timestamp could not be parsed (critical for time features)
ts_missing_before = df['ts'].isna().sum()
df = df[df['ts'].notna()]
print(f"Dropped {ts_missing_before:,} rows due to unparseable timestamps")

# Cities: fill with 'Unknown'
for col in ['user_city', 'merchant_city']:
    df[col] = df[col].fillna('Unknown')

# device_id, ip_address: fill with 'missing_device', 'missing_ip'
df['device_id'] = df['device_id'].fillna('missing_device')
df['ip_address'] = df['ip_address'].fillna('missing_ip')

# account_balance: median per user or global
if 'account_balance' in df.columns:
    df['account_balance'] = df.groupby('user_id')['account_balance'].transform(lambda x: x.fillna(x.median()))
    df['account_balance'] = df['account_balance'].fillna(df['account_balance'].median())

# ─── Step 4: Final missing check & success rate ───
print("\nMissing value report AFTER handling:")
missing_after = df.isna().sum()
print(missing_after[missing_after > 0])

remaining_missing_pct = (df.isna().sum().sum() / (len(df) * len(df.columns))) * 100
print(f"Overall remaining missing values: {remaining_missing_pct:.3f}% of all cells")

success_rate_null = 100 - remaining_missing_pct
print(f"Null handling success rate (non-missing cells): {success_rate_null:.2f}%")


Missing value report BEFORE handling:
                    missing_count  missing_pct
ip_clean                      107         7.39
ip_address                     79         5.46
account_balance                74         5.11
device_id                      73         5.04
merchant_city                  67         4.63
user_city                      55         3.80
transaction_amount             24         1.66
Dropped 0 rows due to unparseable timestamps

Missing value report AFTER handling:
transaction_amount     24
ip_clean              107
dtype: int64
Overall remaining missing values: 0.283% of all cells
Null handling success rate (non-missing cells): 99.72%


In [200]:
# ─── Step 1: Exact row duplicates ───
exact_dup_count_before = df.duplicated().sum()
print(f"Exact duplicate rows before removal: {exact_dup_count_before:,}")

df = df.drop_duplicates(keep='first')
exact_dup_count_after = df.duplicated().sum()

print(f"Exact duplicates removed: {exact_dup_count_before:,}")
print(f"Exact duplicate rows remaining: {exact_dup_count_after:,}")
print(f"Rows after exact dedup: {len(df):,}")

# ─── Step 2: Same transaction_id but different content ───
txid_dup = df['transaction_id'].value_counts()
txid_multi = txid_dup[txid_dup > 1]

print(f"\nTransaction IDs appearing more than once: {len(txid_multi):,}")
print(f"Total rows belonging to repeated transaction_ids: {txid_multi.sum():,}")

if len(txid_multi) > 0:
    print("\nSample conflicting transaction_ids:")
    print(txid_multi.head(8))

# extra rows that should be removed
extra_duplicate_rows = txid_multi.sum() - len(txid_multi)
print(f"Extra duplicate rows to resolve: {extra_duplicate_rows:,}")

# Better conflict strategy: keep most complete row, then latest timestamp
df['non_null_count'] = df.notna().sum(axis=1)
df = df.sort_values(['transaction_id', 'non_null_count', 'ts'])
df = df.groupby('transaction_id').tail(1).drop(columns='non_null_count').reset_index(drop=True)

print(f"\nRows after transaction_id deduplication: {len(df):,}")

remaining_duplicate_txids = df['transaction_id'].duplicated().sum()
print(f"Remaining duplicate transaction_ids: {remaining_duplicate_txids:,}")

if extra_duplicate_rows > 0:
    duplicate_resolution_rate = ((extra_duplicate_rows - remaining_duplicate_txids) / extra_duplicate_rows) * 100
else:
    duplicate_resolution_rate = 100.0

print(f"Duplicate resolution rate: {duplicate_resolution_rate:.2f}%")

Exact duplicate rows before removal: 21
Exact duplicates removed: 21
Exact duplicate rows remaining: 0
Rows after exact dedup: 1,426

Transaction IDs appearing more than once: 0
Total rows belonging to repeated transaction_ids: 0
Extra duplicate rows to resolve: 0

Rows after transaction_id deduplication: 1,426
Remaining duplicate transaction_ids: 0
Duplicate resolution rate: 100.00%


TRAINING

In [276]:
user_stats = df.groupby('user_id')['transaction_amount'].agg(['mean','std','count']).reset_index()
user_stats.columns = ['user_id','user_avg_amount','user_std_amount','user_txn_count']

# Define columns to potentially drop before merge to prevent conflicts
cols_to_drop = [
    'user_avg_amount', 'user_std_amount', 'user_txn_count',
    'amount_vs_user_avg', 'amount_diff', 'amount_zscore'
]

# Drop existing columns if they are present in df
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

# Merge user statistics
df = df.merge(user_stats, on='user_id', how='left')

# Amount deviation features
df['amount_vs_user_avg'] = df['transaction_amount'] / (df['user_avg_amount'] + 1)
df['amount_diff'] = df['transaction_amount'] - df['user_avg_amount']
df['amount_zscore'] = df['amount_diff'] / (df['user_std_amount'] + 1)

In [278]:
df['seconds_since_prev'] = df.groupby('user_id')['ts'].diff().dt.total_seconds().fillna(0)
df['txn_count_last_5min'] = (
    df.groupby('user_id')['seconds_since_prev']
      .transform(lambda x: (x < 300).cumsum())
)

In [203]:
df['hour'] = df['ts'].dt.hour
df['day_of_week'] = df['ts'].dt.dayofweek

df['is_night'] = ((df['hour'] >= 0) & (df['hour'] <= 5)).astype(int)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [204]:
# Device used by multiple users
device_user_count = df.groupby('device_id')['user_id'].nunique()
df['device_user_count'] = df['device_id'].map(device_user_count)

# New device for user
df['is_new_device_for_user'] = (
    df.groupby('user_id')['device_id']
      .transform(lambda x: ~x.duplicated())
).astype(int)

# Number of devices used by user
user_device_count = df.groupby('user_id')['device_id'].nunique()
df['user_device_count'] = df['user_id'].map(user_device_count)

In [205]:
# New city for user
df['is_new_city_for_user'] = (
    df.groupby('user_id')['user_city']
      .transform(lambda x: ~x.duplicated())
).astype(int)

# Location switch detection
df['prev_city'] = df.groupby('user_id')['user_city'].shift(1)

df['location_switch_flag'] = (
    (df['user_city'] != df['prev_city']) &
    df['prev_city'].notna()
).astype(int)

# Number of cities per user
user_city_count = df.groupby('user_id')['user_city'].nunique()
df['user_city_count'] = df['user_id'].map(user_city_count)

In [206]:
# Amount + velocity interaction
df['amount_velocity_interaction'] = df['amount_vs_user_avg'] * df['txn_count_last_5min']

# New device + new location (strong fraud signal)
df['device_location_combo'] = (
    df['is_new_device_for_user'] &
    df['is_new_city_for_user']
).astype(int)

# Night + high amount
df['night_amount_flag'] = (
    (df['is_night'] == 1) &
    (df['amount_vs_user_avg'] > 3)
).astype(int)

In [281]:
df['amount_to_balance_ratio'] = df['transaction_amount'] / (df['account_balance'] + 1)

df['amount_gt_balance'] = (df['transaction_amount'] > df['account_balance']).astype(int)

In [208]:
df

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,...,user_device_count,is_new_city_for_user,prev_city,location_switch_flag,user_city_count,amount_velocity_interaction,device_location_combo,night_amount_flag,amount_to_balance_ratio,amount_gt_balance
0,TXN-00714C5049CD,USR0070,2339.530000,1712068964,Bombay,LKO,Grocery,DEV-2D7B3E2A,mobile,WALLET,...,1,1,NaN,0,2,0.369759,1,0,0.014688,0
1,TXN-00747C38488B,USR0056,1609.794281,20240422042722,hyderabad,BOM,Entertainment,DEV-B2519C05,mobile,NETBANKING,...,4,1,NaN,0,4,0.801428,1,0,0.009563,0
2,TXN-00857C790C38,USR0019,1269.390000,2024-03-02T03:02:09,DEL,delhi,Entertainment,DEV-00E01FDD,web,NETBANKING,...,5,1,NaN,0,3,4.422286,1,1,0.008895,0
3,TXN-00A7AB9C0434,USR0001,235.020000,2024-04-11T20:20:15.611788,ahmedabad,AMD,Healthcare,DEV-CEAEBD15,ATM,CARD,...,2,1,NaN,0,5,0.600173,1,0,0.004284,0
4,TXN-00CEF86EDB60,USR0050,7570.000000,29/06/2024 13:43:36,jaipur,Jaipur,Education,DEV-C5C9E23E,ATM,UNKNOWN,...,3,1,NaN,0,3,1.333080,1,0,0.207218,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1421,TXN-FED4178BBE48,USR0046,5397.930000,20/02/2024 00:10:12,CHENNAI,CHENNAI,Food & Dining,DEV-D544AFBB,mobile,NETBANKING,...,4,0,Chennai,0,4,6.284373,0,0,5397.930000,1
1422,TXN-FED9CE1CA28F,USR0073,7740.180000,2024-03-21T15:53:44,hyderabad,HYDERABAD,Electronics,DEV-441223C2,ATM,CARD,...,5,1,Bengaluru,1,6,15.959477,0,0,7740.180000,1
1423,TXN-FF0B7BE49343,USR0058,5074.171800,2024-06-10T05:02:59.327899,jaipur,JAI,Education,DEV-9A71CA57,mobile,UPI,...,4,0,Kolkata,1,2,4.495377,0,0,0.038787,0
1424,TXN-FF6039B05017,USR0009,3271.898645,2024-02-06T12:53:11.610043,Kolkata,CCU,Grocery,missing_device,web,WALLET,...,4,0,Kolkata,0,2,9.778015,0,0,0.027910,0


In [209]:
df[['amount_vs_user_avg',
    'txn_count_last_5min',
    'is_new_device_for_user',
    'is_new_city_for_user',
    'device_user_count']].describe()

,amount_vs_user_avg,txn_count_last_5min,is_new_device_for_user,is_new_city_for_user,device_user_count
count,1402.000000,1426.000000,1426.000000,1426.000000,1426.000000
mean,0.999530,5.565919,0.194951,0.183029,3.483871
std,1.716083,3.318243,0.396302,0.386826,10.731169
min,0.000000,1.000000,0.000000,0.000000,1.000000
25%,0.492310,3.000000,0.000000,0.000000,1.000000
50%,0.882798,5.000000,0.000000,0.000000,1.000000
75%,1.163503,8.000000,0.000000,0.000000,1.000000
max,26.130805,15.000000,1.000000,1.000000,50.000000


In [210]:
df['amount_vs_user_avg'] = df['amount_vs_user_avg'].fillna(1)
df['txn_count_last_5min'] = df['txn_count_last_5min'].fillna(0)
df['seconds_since_prev'] = df['seconds_since_prev'].fillna(999999)

In [211]:
df['high_amount_flag'] = (df['amount_vs_user_avg'] > 3).astype(int)

In [212]:
df['velocity_flag'] = (df['txn_count_last_5min'] >= 2).astype(int)

In [213]:
df['multi_signal_flag'] = (
    df['high_amount_flag'] +
    df['velocity_flag'] +
    df['is_new_device_for_user'] +
    df['is_new_city_for_user']
)

STEP2: Training

In [214]:
# Core signals
df['high_amount_flag'] = (df['amount_vs_user_avg'] > 3).astype(int)
df['extreme_amount_flag'] = (df['amount_vs_user_avg'] > 6).astype(int)

df['velocity_flag'] = (df['txn_count_last_5min'] >= 2).astype(int)
df['fast_txn_flag'] = (df['seconds_since_prev'] < 60).astype(int)

# Combination already built
# df['multi_signal_flag'] exists (sum of signals)

# IP / device already built
# ip_multi_user_flag, ip_anomaly, device_user_count, is_new_device_for_user, is_new_city_for_user

# Score (weighted)
df['fraud_score'] = (
    3*df['extreme_amount_flag'] +
    2*df['high_amount_flag'] +
    2*df['velocity_flag'] +
    1*df['fast_txn_flag'] +
    2*df['is_new_device_for_user'] +
    2*df['is_new_city_for_user'] +
    2*df['device_location_combo'] +      # very strong signal
    2*df['ip_multi_user_flag'] +
    1*df['ip_anomaly'] +
    1*(df['device_user_count'] > 5).astype(int) +
    1*(df['amount_to_balance_ratio'] > 0.7).astype(int)
)

In [215]:
# Inspect distribution
df['fraud_score'].describe()
df['fraud_score'].value_counts().sort_index()

# Start with this threshold
THRESHOLD = 5

df['fraud_label'] = (df['fraud_score'] >= THRESHOLD).astype(int)

# Check balance
fraud_rate = df['fraud_label'].mean()
print("Fraud rate:", fraud_rate)

Fraud rate: 0.23492286115007013


In [216]:
df.groupby('fraud_label')[[
    'amount_vs_user_avg',
    'txn_count_last_5min',
    'is_new_device_for_user',
    'is_new_city_for_user'
]].mean()

,amount_vs_user_avg,txn_count_last_5min,is_new_device_for_user,is_new_city_for_user
fraud_label,,,,
0,0.827943,5.993584,0.054079,0.038497
1,1.558374,4.173134,0.653731,0.653731


In [217]:
df['very_high_risk'] = (df['fraud_score'] >= 7).astype(int)

In [218]:
df['strong_combo_flag'] = (
    (df['is_new_device_for_user'] == 1) &
    (df['is_new_city_for_user'] == 1)
).astype(int)

In [219]:
df['fraud_score'] += 3 * df['strong_combo_flag']

In [220]:
THRESHOLD = 6   # slightly stricter now

df['fraud_label'] = (df['fraud_score'] >= THRESHOLD).astype(int)

print("New fraud rate:", df['fraud_label'].mean())

New fraud rate: 0.135343618513324


STEP 3: TRAINING

In [221]:
from sklearn.ensemble import IsolationForest

iso_features = [
    'transaction_amount',
    'amount_vs_user_avg',
    'txn_count_last_5min',
    'seconds_since_prev'
]

X_iso = df[iso_features].fillna(0)

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

iso.fit(X_iso)

# -1 = anomaly → convert to 1
df['anomaly_score'] = (iso.predict(X_iso) == -1).astype(int)

In [269]:
FEATURES = [
    'amount_vs_user_avg',
    'amount_zscore',
    'txn_count_last_5min',
    'seconds_since_prev',
    'is_new_device_for_user',
    'is_new_city_for_user',
    'device_user_count',
    'user_device_count',
    'location_switch_flag',
    'user_city_count',
    'ip_multi_user_flag',
    'ip_anomaly',
    'ip_is_private',
    'hour',
    'is_night',
    'amount_to_balance_ratio',
    'device_location_combo',
    'amount_velocity_interaction',
    'strong_combo_flag',
    'anomaly_score',
    'rolling_avg_3',
    'spike_flag'
]

X = df[FEATURES].fillna(0)
y = df['fraud_label']

KeyError: "['rolling_avg_3', 'spike_flag'] not in index"

In [223]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced'   # important for fraud
)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 154, number of negative: 986
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000561 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1351
[LightGBM] [Info] Number of data points in the train set: 1140, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=300)

In [224]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       247
           1       0.95      0.92      0.94        39

    accuracy                           0.98       286
   macro avg       0.97      0.96      0.96       286
weighted avg       0.98      0.98      0.98       286



In [225]:
df['fraud_prob'] = model.predict_proba(X)[:, 1]

In [226]:
df['fraud_prob'].describe()

,fraud_prob
count,1.426000e+03
mean,1.346824e-01
std,3.411247e-01
min,1.430724e-07
25%,2.097101e-07
50%,5.579584e-07
75%,4.173480e-05
max,9.999997e-01


In [227]:
model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=5,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=0.5,
    class_weight='balanced',
    random_state=42
)

In [228]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [229]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=5,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=0.5,
    class_weight='balanced',
    random_state=42
)

In [230]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 154, number of negative: 986
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000692 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1351
[LightGBM] [Info] Number of data points in the train set: 1140, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.05, max_depth=5, min_child_samples=30,
               n_estimators=300, num_leaves=15, random_state=42, reg_alpha=0.5,
               reg_lambda=0.5, subsample=0.8)

In [231]:
df['fraud_prob'] = model.predict_proba(X)[:,1]

In [232]:
df['fraud_prob'].describe()

,fraud_prob
count,1426.000000
mean,0.141840
std,0.337060
min,0.000016
25%,0.000067
50%,0.000161
75%,0.016937
max,0.999677


STEP4:TRAINING

In [233]:
df['fraud_stage1'] = df['fraud_prob'] > 0.3

In [234]:
df['final_fraud'] = df['fraud_stage1'] & (
    (df['fraud_score'] >= 6) |
    (df['strong_combo_flag'] == 1) |
    (df['anomaly_score'] == 1)
)

In [235]:
def fraud_type(row):
    if row['strong_combo_flag'] == 1:
        return "Device + Location Fraud"
    elif row['amount_vs_user_avg'] > 5:
        return "High Amount Fraud"
    elif row['txn_count_last_5min'] >= 3:
        return "Velocity Fraud"
    elif row['is_new_device_for_user']:
        return "Device Fraud"
    elif row['is_new_city_for_user']:
        return "Location Fraud"
    else:
        return "Mixed Fraud"

df['fraud_type'] = df.apply(
    lambda x: fraud_type(x) if x['final_fraud'] == 1 else "Not Fraud",
    axis=1
)

In [236]:
from sklearn.metrics import classification_report

print(classification_report(y, df['final_fraud']))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1233
           1       0.98      0.99      0.99       193

    accuracy                           1.00      1426
   macro avg       0.99      1.00      0.99      1426
weighted avg       1.00      1.00      1.00      1426



STEP5: TRAINING

In [237]:
final_df = df[[
    'user_id',
    'transaction_amount',
    'ts',
    'device_id',
    'user_city',
    'fraud_prob',
    'final_fraud',
    'fraud_type'
]].copy()

In [238]:
final_df['fraud_label'] = final_df['final_fraud'].map({
    0: "Not Fraud",
    1: "Fraud"
})

In [239]:
total_txn = len(final_df)
fraud_count = final_df['final_fraud'].sum()
fraud_rate = fraud_count / total_txn

print("Total Transactions:", total_txn)
print("Fraud Count:", fraud_count)
print("Fraud Rate:", fraud_rate)

Total Transactions: 1426
Fraud Count: 195
Fraud Rate: 0.13674614305750352


In [240]:
fraud_type_counts = final_df['fraud_type'].value_counts()
print(fraud_type_counts)

fraud_type
Not Fraud                  1231
Device + Location Fraud     122
Velocity Fraud               46
High Amount Fraud            18
Device Fraud                  6
Location Fraud                3
Name: count, dtype: int64


In [241]:
top_fraud = final_df.sort_values('fraud_prob', ascending=False).head(10)
print(top_fraud)

    user_id  transaction_amount                         ts       device_id  \
26  USR0038         5930.192200 2024-02-04 04:15:19.325234  missing_device   
11  USR0046         5487.559802 2024-02-19 12:29:11.924128    DEV-D544AFBB   
60  USR0073         8250.870000 2024-05-20 18:50:00.000000    DEV-441223C2   
9   USR0076         7131.210000 2024-05-21 21:55:44.055036    DEV-85FA2A2D   
97  USR0053         2063.043512 2024-05-16 06:26:00.000000  missing_device   
48  USR0077         1647.499394 2024-03-13 12:16:27.114601  missing_device   
24  USR0055         2538.920037 2024-03-27 14:36:00.000000  missing_device   
13  USR0060         7962.863854 2024-02-12 22:20:00.000000    DEV-D8B7F0E3   
95  USR0072         8611.492700 2024-03-06 09:36:57.000000  missing_device   
85  USR0065         4328.888836 2024-04-03 22:58:30.000000  missing_device   

    user_city  fraud_prob  final_fraud               fraud_type fraud_label  
26    Lucknow    0.999677         True  Device + Location Fraud

In [242]:
fraud_city = final_df[final_df['final_fraud'] == 1]['user_city'].value_counts()
print(fraud_city)

user_city
Jaipur       29
Mumbai       21
Pune         17
Hyderabad    16
Chennai      16
Ahmedabad    15
Unknown      15
Lucknow      15
Delhi        13
Kolkata      13
Bengaluru    13
Dubai         2
Singapore     2
New York      2
Hyderab#      1
Hyde          1
Hyder...      1
Madr#         1
New Yor#      1
Beng...       1
Name: count, dtype: int64


In [243]:
final_df['date'] = final_df['ts'].dt.date

fraud_time = final_df.groupby('date')['final_fraud'].sum()
print(fraud_time)

date
2024-01-01    2
2024-01-02    0
2024-01-03    0
2024-01-04    0
2024-01-05    0
             ..
2024-12-02    0
2024-12-03    0
2024-12-04    1
2024-12-05    0
2024-12-06    1
Name: final_fraud, Length: 215, dtype: int64


In [244]:
final_df.to_csv("fraud_detection_output.csv", index=False)

STEP6: TRAINING

In [245]:
from fastapi import FastAPI
import pandas as pd

app = FastAPI()

@app.post("/predict")
def predict(file):
    df = pd.read_csv(file.file)

    # pipeline
    # model

    return {"fraud_count": int(df['final_fraud'].sum())}

In [246]:
df['fraud_stage1'] = df['fraud_prob'] > THRESHOLD

In [247]:
df['final_fraud'] = df['fraud_stage1'] & (
    (df['fraud_score'] >= 4) |
    (df['txn_count_last_5min'] >= 4) |
    (df['anomaly_score'] == 1)
)

STEP7:TRAINING

In [248]:
def fraud_type(row):
    if row['amount_vs_user_avg'] > 5:
        return "High Amount Fraud"

    elif row['txn_count_last_5min'] >= 3:
        return "Velocity Fraud"

    elif row['is_new_device_for_user']:
        return "Device Fraud"

    elif row['is_new_city_for_user']:
        return "Location Fraud"

    elif row['ip_multi_user_flag']:
        return "Shared Device/IP Fraud"

    else:
        return "Mixed Fraud"

df['fraud_type'] = df.apply(fraud_type, axis=1)

In [249]:
total_txn = len(df)
fraud_count = df['final_fraud'].sum()
fraud_rate = fraud_count / total_txn

print("Total Transactions:", total_txn)
print("Fraud Detected:", fraud_count)
print("Fraud Rate:", fraud_rate)

Total Transactions: 1426
Fraud Detected: 0
Fraud Rate: 0.0


In [250]:
total_txn = len(df)
fraud_count = df['final_fraud'].sum()
fraud_rate = fraud_count / total_txn

print("Total Transactions:", total_txn)
print("Fraud Detected:", fraud_count)
print("Fraud Rate:", fraud_rate)

Total Transactions: 1426
Fraud Detected: 0
Fraud Rate: 0.0


In [251]:
df['fraud_type'].value_counts()

,count
fraud_type,
Velocity Fraud,1106
Device Fraud,166
Mixed Fraud,117
High Amount Fraud,19
Location Fraud,18


In [252]:
df.sort_values('fraud_prob', ascending=False).head(10)

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,...,fast_txn_flag,fraud_score,fraud_label,very_high_risk,strong_combo_flag,anomaly_score,fraud_prob,fraud_stage1,final_fraud,fraud_type
26,TXN-05AAA195C0E9,USR0038,5930.192200,2024-04-02T04:15:19.325234,lucknow,Madras,Entertainment,missing_device,mobile,CARD,...,1,12,1,1,1,0,0.999677,False,False,Device Fraud
11,TXN-02E7C6E5C78F,USR0046,5487.559802,2024-02-19T12:29:11.924128,Madras,Madras,Entertainment,DEV-D544AFBB,mobile,NETBANKING,...,1,11,1,1,1,0,0.999654,False,False,Device Fraud
60,TXN-0CAD73EEE929,USR0073,8250.870000,05-20-2024 18:50,bangalore,ahmedabad,Education,DEV-441223C2,ATM,CARD,...,1,11,1,1,1,0,0.999636,False,False,Device Fraud
9,TXN-0211EFEBABDA,USR0076,7131.210000,2024-05-21T21:55:44.055036,Bangalore,Bangalore,Travel,DEV-85FA2A2D,web,NETBANKING,...,1,11,1,1,1,0,0.999623,False,False,Device Fraud
97,TXN-12BF4B20CC33,USR0053,2063.043512,"May 16, 2024 06:26 AM",DEL,New Delhi,Electronics,missing_device,mobile,CARD,...,1,11,1,1,1,0,0.999617,False,False,Device Fraud
48,TXN-09C8427358C3,USR0077,1647.499394,2024-03-13T12:16:27.114601,PNQ,Pune,Clothing,missing_device,ATM,UPI,...,1,13,1,1,1,0,0.999612,False,False,Device Fraud
24,TXN-05309F3D78C8,USR0055,2538.920037,"March 27, 2024 02:36 PM",HYD,HYDERABAD,Electronics,missing_device,mobile,UPI,...,1,11,1,1,1,0,0.999607,False,False,Device Fraud
13,TXN-02F02312E2C5,USR0060,7962.863854,"February 12, 2024 10:20 PM",PNQ,PNQ,Grocery,DEV-D8B7F0E3,web,UPI,...,1,11,1,1,1,0,0.999603,False,False,Device Fraud
95,TXN-12A1FEFF0726,USR0072,8611.492700,20240306093657,jaipur,jaipur,Education,missing_device,ATM,WALLET,...,1,12,1,1,1,0,0.999603,False,False,Device Fraud
85,TXN-11650A26F88C,USR0065,4328.888836,03/04/2024 22:58:30,BOM,mUMBAI,Electronics,missing_device,web,NETBANKING,...,1,12,1,1,1,0,0.999596,False,False,Device Fraud


In [253]:
df.to_csv("fraud_detection_output.csv", index=False)

In [254]:
predicted_fraud = df['final_fraud'].sum()
total_txn = len(df)

print("Predicted Fraud Count:", predicted_fraud)
print("Total Transactions:", total_txn)
print("Fraud Rate:", predicted_fraud / total_txn)

Predicted Fraud Count: 0
Total Transactions: 1426
Fraud Rate: 0.0


In [255]:
predicted_fraud = df['final_fraud'].sum()
total_txn = len(df)

print("Predicted Fraud Count:", predicted_fraud)
print("Total Transactions:", total_txn)
print("Fraud Rate:", predicted_fraud / total_txn)

Predicted Fraud Count: 0
Total Transactions: 1426
Fraud Rate: 0.0


In [256]:
df['fraud_prob'].describe()

,fraud_prob
count,1426.000000
mean,0.141840
std,0.337060
min,0.000016
25%,0.000067
50%,0.000161
75%,0.016937
max,0.999677


In [257]:
df['fraud_stage1'] = df['fraud_prob'] > 0.3
print("Stage 1 frauds:", df['fraud_stage1'].sum())

Stage 1 frauds: 196


In [258]:
df['final_fraud'] = df['fraud_stage1'] & (
    (df['fraud_score'] >= 7) |
    (
        (df['strong_combo_flag'] == 1) &
        (df['fraud_score'] >= 6)
    ) |
    (
        (df['anomaly_score'] == 1) &
        (df['fraud_score'] >= 6)
    )
)

In [259]:
print("Stage 2 frauds:", df['final_fraud'].sum())

Stage 2 frauds: 153


In [260]:
print("Stage 1:", df['fraud_stage1'].sum())
print("Final Fraud:", df['final_fraud'].sum())

Stage 1: 196
Final Fraud: 153


In [261]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']

In [262]:
final_df

,user_id,transaction_amount,ts,device_id,user_city,fraud_prob,final_fraud,fraud_type,fraud_label,date
0,USR0070,2339.530000,2024-04-02 14:42:44.000000,DEV-2D7B3E2A,Mumbai,0.998891,True,Device + Location Fraud,NaN,2024-04-02
1,USR0056,1609.794281,2024-04-22 04:27:22.000000,DEV-B2519C05,Hyderabad,0.998713,True,Device + Location Fraud,NaN,2024-04-22
2,USR0019,1269.390000,2024-02-03 03:02:09.000000,DEV-00E01FDD,Delhi,0.998779,True,Device + Location Fraud,NaN,2024-02-03
3,USR0001,235.020000,2024-11-04 20:20:15.611788,DEV-CEAEBD15,Ahmedabad,0.998287,True,Device + Location Fraud,NaN,2024-11-04
4,USR0050,7570.000000,2024-06-29 13:43:36.000000,DEV-C5C9E23E,Jaipur,0.999388,True,Device + Location Fraud,NaN,2024-06-29
...,...,...,...,...,...,...,...,...,...,...
1421,USR0046,5397.930000,2024-02-20 00:10:12.000000,DEV-D544AFBB,Chennai,0.000706,False,Not Fraud,NaN,2024-02-20
1422,USR0073,7740.180000,2024-03-21 15:53:44.000000,DEV-441223C2,Hyderabad,0.988897,True,Velocity Fraud,NaN,2024-03-21
1423,USR0058,5074.171800,2024-10-06 05:02:59.327899,DEV-9A71CA57,Jaipur,0.000095,False,Not Fraud,NaN,2024-10-06
1424,USR0009,3271.898645,2024-06-02 12:53:11.610043,missing_device,Kolkata,0.001174,False,Not Fraud,NaN,2024-06-02


In [263]:
from sklearn.model_selection import train_test_split

X = df[FEATURES].fillna(0)
y = df['fraud_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [264]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 154, number of negative: 986
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000312 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1351
[LightGBM] [Info] Number of data points in the train set: 1140, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.05, max_depth=5, min_child_samples=30,
               n_estimators=300, num_leaves=15, random_state=42, reg_alpha=0.5,
               reg_lambda=0.5, subsample=0.8)

In [265]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9825174825174825
Precision: 0.9047619047619048
Recall: 0.9743589743589743
F1 Score: 0.9382716049382716


In [266]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.9980276134122288


In [267]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']

In [268]:
from google.colab import files

files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task
Update the fraud detection model's features by adding 'iso_score', 'rolling_avg_3', 'spike_flag', and 'ip_multi_user_flag' to the existing `FEATURES` list. Then, implement an ensemble model by training a `RandomForestClassifier` alongside the current `LightGBMClassifier` using the updated feature set. Generate probability predictions from both models and combine them into a new 'ensemble_fraud_prob' column using a weighted average of 0.6 for LightGBM and 0.4 for Random Forest. Set a prediction threshold of 0.4 for 'ensemble_fraud_prob' to derive the 'final_fraud' predictions, and evaluate the ensemble model's performance using accuracy, precision, recall, and F1 score. Finally, provide a summary of the overall improvements achieved by this hybrid fraud detection system.

## Update Model Features

### Subtask:
Modify the 'FEATURES' list in the notebook to incorporate the newly generated features: 'iso_score' (from Isolation Forest), 'rolling_avg_3', 'spike_flag' (from sequence features), and 'ip_risk_flag' (from IP risk analysis). This ensures these new signals are used in the model training process.


# Task
Update the fraud detection model's features by including 'iso_score', 'rolling_avg_3', 'spike_flag', and 'ip_risk_flag' (using 'ip_multi_user_flag' if 'ip_risk_flag' is not present) in the `FEATURES` list. Then, create an ensemble model by training both a `LightGBMClassifier` and a `RandomForestClassifier` with the refined feature set. Combine their predictions with weights (0.6 for LightGBM, 0.4 for RandomForest) into an 'ensemble_fraud_prob' column. Finally, generate 'final_fraud' predictions using a 0.4 threshold on 'ensemble_fraud_prob' and evaluate the ensemble model's performance by calculating accuracy, precision, recall, and F1 score.

## Ensure Feature Creation

### Subtask:
Ensure all necessary features, including 'iso_score', 'rolling_avg_3', 'spike_flag', and 'ip_multi_user_flag', as well as 'amount_vs_user_avg', 'amount_zscore', 'seconds_since_prev', 'txn_count_last_5min', and 'amount_to_balance_ratio', are correctly computed and added to the DataFrame `df`.


# Task
```python
df['rolling_avg_3'] = df.groupby('user_id')['transaction_amount'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['spike_flag'] = (df['transaction_amount'] > 2 * df['rolling_avg_3']).astype(int)
```

## Ensure Rolling Avg and Spike Flag Features

### Subtask:
Execute the cell that creates 'rolling_avg_3' and 'spike_flag' based on user transaction history.
